<a href="https://colab.research.google.com/github/Edem-m/JaxGitWorkship/blob/readme_Edem/BDSiC_Day_5_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GWAS QC Pipeline
- General DS issues:
  1. Removing inaccurate data -- context dependent
  2. Identifying outliers -- exploratory visualization
  3. Addressing missing data -- we will work through various examples of this
  4. Removing duplicated data -- we work through an example of this
  5. Standardizing data formats: remove unnecessary columns, convert data types
  6. Ensuring that your data fits the assumptions of your model

---
- Review: Specific GWAS issues:
  1. Shared ancestry
  2. LD: association not causation

---

(From BDSiC_5A)
Each particular dataset and question will develop a pipeline to address unique challenges that arise from unique aspects of the data/question. Genomic databases are a great example of a specialized pipeline to reduce the bias of non-independence on top of the more typical missing or inaccurate data grooming:

       a. removing rare or monomorphic variants - minor allele frequency
       b. filtering missing SNPs
       c. identifying and removing genotyping errors
           * heterozygosity
           * sex discrepancy
           * removing variants that are not in Hardy-Weinberg equilibrium (which can indicate: genotyping errors, batch effects, population stratification)
           * PCA is straightforward way to identify population stratification or batch effects
       d. accounting for ancestry and relatedness - another way population stratification pops up! Cases and controls should be matched by ancestry to avoid confounding and, therefore, false positives.
       e. account for false positives (you are testing millions of hypotheses simultaneously)  

    * The program Plink is standard for analysis GWAS and it requires files of specific formats: bed, fam, bim files all contain slightly different information, including gneomic variants, and family relationships. Plink makes the quality control step easier (a huge challenge) with built-in functions for the above. There are python packages that wrap around Plink and multivariate analysis tools to create a seamless pipeline.

# representing the 3-D world in 2-D: Lessons of the Mercator projection
Benefits of Mercator projection on maps:
* Rhumb lines are straight so navigation is easier
Cons:
* Northern land masses look HUGE and Southern land masses look tiny

More accurate:
* https://www.discovery.com/science/AuthaGraph-World-Accurate-Map
* Gall-Peters projection

Think about Subway maps? Are they accurate representations of the cityscape or are they not (and if not, why do they exist?)

* __There are ALWAYS DELIBERATE decisions that we make about what to emphasize__

# Data will speak for itself, when it cleans itself.
- What is "tidy data"? https://vita.had.co.nz/papers/tidy-data.pdf
- Adopting these principles when creating your own datasets will save you so much time. However, we are usually using someone else's data and they probably didn't adopt these principles.
- Nicely bulleted principles here: https://kbroman.org/dataorg/

# General Principles of Data Cleaning, Data Preparation & QC:
* The secret shame of Data Science: Data preparation is tedious, requires context, requires coding skills, demands creativity, doesn't feel like progress :(
* DOCUMENT, DOCUMENT, DOCUMENT your process
  - NOT just WHAT, but WHY. You're not going to remember the justifications tomorrow, let alone in a week or month or year. Help out your future self!
  - no one is 'good' at this, but it is crucial.
  - Trust no-one, not even (past) yourself -- (future) self
---
## Fundamentals
1. Removing inaccurate data
2. Identifying outliers
3. Addressing missing data
4. Removing duplicated data
5. Standardizing data formats (especially dates!): remove unnecessary columns, convert data types, check merged columns/rows
6. Ensuring that your data fits the assumptions of your model
---
## Questions:
1. Is this data what I expect, even when I consider things that could have gone wrong in the data collection process? What data patterns would be present, if various common things had gone wrong?
2. Have I checked my calculations?
3. Can I explain outliers/oddities? Context matters: these could be real, important values, or they could be artifacts, or they could be data entry mistakes.
4. Does my process/pipeline make sense on a broad scale?


In [2]:
# typical Data Cleaning stack
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,MinMaxScaler,OrdinalEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer

In [4]:
# Example 1: we will use this to see two approaches to missing data.
# mostly follows this: https://github.com/sumony2j/Data_Cleaning_Preprocessing/blob/main/Sample_Data_Cleaning%26Preprocessing.ipynb

#df1 = pd.read_csv('./Simple_Data_Ex1.csv')
#df1
# This is a small data set so we can visually see if there are any outliers or 'strange' data.
# however, since we can't talk to the creators of this data set it is challenging to determine accuracy, outside
# of is the data generally 'reasonable'?

a = 5
b = 3
a+b

print(a+b)

8


In [ ]:
# how much missing data is there in each of these columns?
# How do you convert all msising data into NAN in the first firstplace?  It's not like during the data collection the put NAN there.
df1.isnull().sum()

In [ ]:
# how do we want to deal with missing data? There are a handful of common strategies:
# 0, distinctive value (-999), or mean of the column
# ---------
# 1. A solid choice is to replace the NaN with 0.0
# we can do that with the SimpleImputer that we imported from the sklearn.impute module:
Imputer1 = SimpleImputer(missing_values=np.nan,strategy='constant',fill_value=0)
# apply the results to the df1 but only on columns that have missing data as counted in the cell above:
#columns 1 and 2.
df1.iloc[:,1:3]=Imputer1.fit_transform(df1.iloc[:,1:3])

In [ ]:
# let's check in again on our dataframe
df1

In [ ]:
#2. mean of the column
# note: since the data is no longer missing as we just modified it above to be 0, we need to
# re-read in the data set to get this to work.
df1b = pd.read_csv('./Simple_Data_Ex1.csv')
df1b
Imputer2=SimpleImputer(missing_values=np.nan,strategy='mean')
df1b.iloc[:,1:3]=Imputer2.fit_transform(df1b.iloc[:,1:3])

In [ ]:
# let's check in again:
df1b
round(df1b,2)

In [ ]:
# Here is a second dataset. It is a bit larger than the last example.
# taken from here: https://github.com/sumony2j/Data_Cleaning_Preprocessing/blob/main/Airbnb_NY_Data_Cleaning%26Preprocessing.ipynb
df2 = pd.read_csv('./Data_example2.csv')
# we just want to peak at the data - since it is large
df2.head()

In [ ]:
# how big is the data?
df2.shape
# Notice the difference between an attribute -- like shape -- and a method?
df2.info()

In [ ]:
# let's see how many missing data pieces there are in each column.
df2.isnull().sum()

In [ ]:
# some of these columns are not useful to answer our particular question(s).
# we can get rid of columns like so:
df2.drop(['name','host_name','last_review'],axis=1,inplace=True)

In [ ]:
#let's peek again:
print(df2.head())
# see where any remaining missing data is in the columns that we decided to keep
df2.isnull().sum()

In [ ]:
# instead of mean, median or 0.0 value, we could replace mssing values with the most frequent value
imputer = SimpleImputer(missing_values=np.nan,strategy='most_frequent')
# or we could use the strategy='constant',fill_value=0
df2[['reviews_per_month']]=imputer.fit_transform(df2[['reviews_per_month']])
df2.head(25)


In [ ]:
neighbourhood=pd.DataFrame(df2['neighbourhood'])
neighbourhood

In [ ]:
df2_unique_neighbourhoods=df2["neighbourhood"].unique()
df2_unique_neighbourhoods

In [ ]:
#https://github.com/durgenious/loan_data_processing
# use data loan.csv
df3 = pd.read_csv('loan.csv')  # load 'loan.csv'
df3.head(5) # displays first 5 records in the DataFrame
# df3.tail(5) # displays last 5 records in the DataFrame

In [ ]:
print(df3.info())
print("------")
print(df3.describe())
print("~~~~~~")
print(df3.shape)
print("**********")
print(df3.isnull().sum())

In [ ]:
# drop duplicate rows
dup = df3.duplicated(keep='first')   # Find duplicate rows in the DataFrame 'df' based on all columns, keeping the first occurrence.
dropped = df3.loc[dup]   # DataFrame 'dropped' containing only the duplicated rows.
dropped['UID'].unique()[0]    # Extract the unique identifier ('UID') of the duplicated rows.

In [ ]:
df3.loc[df3['UID'].isin(dropped['UID'].unique())]   # Filter the original DataFrame to show only rows with the same 'UID' as the duplicated rows.

In [ ]:
df3.drop_duplicates(inplace=True)  # drop duplicate rows from the original DataFrame
df3.shape  # shape of the DataFrame after dropping duplicates (1 duplicate)

In [ ]:
dup2 = df3.duplicated(subset=['UID'], keep='first')    # Find duplicate rows in the DataFrame 'df' based on 'UID', keeping the first occurrence.
dropped2 = df3.loc[dup2]
dropped2['UID'].unique()[0]   # Extract the unique identifier ('UID') of the duplicated rows.

In [ ]:
df3.loc[df3['UID'].isin(dropped2['UID'].unique())]    # Filter the original DataFrame to show only rows with the same 'UID' as the duplicated rows.

In [ ]:
df3.drop_duplicates('UID', keep='first', inplace=True)  # drop duplicates('Column', keep='first')
df3.shape    # shape of the DataFrame after dropping duplicates (2 duplicates)

In [ ]:
# Data standardization
df3['Marital_status'].value_counts() # counts the occurrences of each unique value

In [ ]:
df3['Marital_status'] = df3['Marital_status'].str.upper()   # converts all the values to Uppercase
df3['Marital_status'].value_counts()   # counts the occurrences of 'YES' and 'NO' after case conversion

In [ ]:
df3['Sex'].value_counts()    # counts the occurrences of each unique value

In [ ]:
# We replace 'M' -> 'Male' and 'F' -> 'Female'
df3['Sex'] = df3['Sex'].replace({'M' : 'Male', 'F' : 'Female'})    # replace({old_value : new_value}) dict(old_value=new_value)
df3['Sex'].value_counts()    # counts the occurrences of 'Male' and 'Female' after replacement

In [ ]:
# Now that we have familiarized ourselves with the data, maybe we can tackle inaccurate records?
print(df3.describe())   # Checking for Incorrect Records
print("~~~~~~~~~~~~~~~~~~")
print(df3.shape[0])   # No of rows
print("~~~~~~~~~~~~~~~~~~")
df3.isnull().sum()

There are two suspicious things going on with the Age column that speaks to both accuracy and missing data. Who is -12 years old? How is that possible? Plus there are 6 Ages that we don't have at all.

The strategy for Age might be to replace the missing data and the inaccurate date (let's say any age under 10) with the mean value of the other values in that column. This strategy is worked below:

In [ ]:
# Why is there a minimum age tht is -12? Who is -12 years old? We will need to remove this:
print(df3['Age'].min())   # Incorrect Minimum age
# we will replace all ages that are < 10 with a missing value and then we will address missing values in the next cell
df3.loc[df3["Age"]<10,"Age"]=df3["Age"].mean()
#check to see the min of the Age column
df3['Age'].min()
# That's better! Let's replace the 6 missing values with the mean value of the column, too.
#df3['Age'].fillna(df3['Age'].mean(),inplace=True)
df3["Age"] = df3["Age"].fillna(df3["Age"].mean())
# You can also use the SimpleImputer function from sklearn
# as we used in the df1, df2 examples.

In [ ]:
print(df3["Age"])
# have we replaced all the missing elements in the Age column with the mean?
df3["Age"].isnull().sum()

In [ ]:
# handle missing values:
print(df3.shape)
print("--------------")
df3.isnull().sum()

In [ ]:
# replace missing (null or NaN) values with mean of that column
df3['Loan_amount'].fillna(df3['Term_months'].mean())
#df3.isnull().sum()
df3['Term_months'].fillna(df3['Term_months'].mean())
print("*****************")
df3.isnull().sum()

In [ ]:
# we might decide that some of the columns contain irrelevant information
# so we can just drop all the rows with missing values
df3.dropna(inplace=True)   # We drop the records with missing values
print(df3.isnull().sum())   # count missing values
df3.shape # we can see how many rows we have now dropped.